### BFS

Suchgraph ist bereits ein Baum.
Brauchen also nicht zu Prüfen, ob eine Knoten bereits besucht wurde.
Da in einem Baum jeder Knoten höchsten einen Elternknoen hat, brauchen wir auch keine Hilfe zum Finden des 
Wegs nach Hause.

<img src='/files/images/graph_AJ.png' width='300'>  

cut into a tree

<img src='/files/images/treeD_graph_AJ.png'>  

<img src='/files/images/treeG_graph_AJ.png'>  

<img src='/files/images/treeF_graph_AJ.png'>  

<img src='/files/images/treeH_graph_AJ.png'>  

In [3]:
# Unweighted graph with 10 nodes (A through J)
graph = {
    'A': ('B', 'C', 'D'),
    'B': ('A', 'E', 'F'),
    'C': ('A', 'G'),
    'D': ('A', 'H'),
    'E': ('B', 'I'),
    'F': ('B', 'J'),
    'G': ('C', 'J'),
    'H': ('D',),
    'I': ('E',),
    'J': ('F', 'G')
}


&uuml;

In [4]:
from collections import deque


def search_bf(start, graph, target=None):
    node_to_visite = deque([start])
    go_back = {start: None}

    while node_to_visite:
        node = node_to_visite.pop()
        if target and node == target:
            return node, go_back

        for neighbor in graph.get(node, ()):
            if neighbor in go_back:
                continue
            go_back[neighbor] = node
            node_to_visite.appendleft(neighbor)


def get_path_home(node, go_back):
    path = []
    while node:
        path.append(node)
        node = go_back[node]
    return path

In [5]:
# kuerzester Weg von D nach F
target, child_parent = search_bf('D', graph, 'F')
get_path_home(target, child_parent)

['F', 'B', 'A', 'D']

In [20]:
def draw_tree(canvas, tree, x, y, xmin, xmax, dy, radius):
    val, children = tree

    # der einfache Fall
    canvas.fill_text(f'{val}', x, y)
    canvas.stroke_circle(x, y, radius)
    if not children:
        return


    # Rekursion
    dx = (xmax-xmin) // len(children)
    for i, child in enumerate(children):
        cx, cy = xmin + i*dx + dx//2, y + dy
        canvas.stroke_line(x, y+radius, cx, cy-radius)
        draw_tree(canvas,
                  child,
                  x=cx,
                  y=cy,
                  xmin=cx-dx//2,
                  xmax=cx+dx//2,
                  dy=dy,
                  radius=radius,
                  )

In [54]:
import widget_helpers as W


canvas = W.get_canvas(300, 200)
canvas.text_align = 'center'
canvas.text_baseline = 'middle'
canvas

Canvas(height=200, layout=Layout(border_bottom='1px solid black', border_left='1px solid black', border_right=…

In [55]:
canvas.clear()
draw_tree(canvas, tree, x=150, y=10, xmin=0, xmax=300, dy=30, radius=8)

In [26]:
def cp2rpcs(cp):
    pcs = {}
    root = None
    for k, v in cp.items():
        if v is None:
            root = k
            continue
        pcs.setdefault(v, []).append(k)
    return root, pcs


def rpcs2tree(root, pcs):
    if root not in pcs:
        return (root, [])

    children = [rpcs2tree(child, pcs) for child in pcs[root]]
    return (root, children)    

In [52]:
root, pcs = cp2rpcs(child_parent)
tree = rpcs2tree(root, pcs)
tree

('H',
 [('D',
   [('A',
     [('B', [('E', [('I', [])]), ('F', [('J', [])])]), ('C', [('G', [])])])])])

In [2]:
import sokoban_graph as S

In [9]:
goal, child_parent = search_bf((S.boxes, S.player), lambda x: S.graph.get(x,()), lambda x: x[0] == S.targets)

In [10]:
goal

(frozenset({2, 3, 4, 6, 9}), 13)

In [23]:
path = get_path_home(goal, child_parent)
path_to_target = [x[0] for x in reversed(path)]
path_to_target

[frozenset({3, 6, 9, 13, 16}),
 frozenset({3, 6, 9, 16, 19}),
 frozenset({3, 6, 9, 12, 16}),
 frozenset({2, 3, 9, 12, 16}),
 frozenset({3, 7, 9, 12, 16}),
 frozenset({3, 7, 8, 9, 12}),
 frozenset({2, 3, 8, 9, 12}),
 frozenset({3, 6, 8, 9, 12}),
 frozenset({3, 6, 9, 12, 13}),
 frozenset({1, 6, 9, 12, 13}),
 frozenset({1, 5, 6, 9, 13}),
 frozenset({1, 3, 5, 9, 13}),
 frozenset({3, 4, 5, 9, 13}),
 frozenset({2, 3, 4, 9, 13}),
 frozenset({2, 3, 4, 6, 9})]

In [30]:
def get_moves(path):
    moves = []
    for a, b in zip(path[:-1], path[1:]):
        move = set((a - b)).pop(), set((b - a)).pop()
        moves.append(move)
    return moves

In [32]:
moves = get_moves(path_to_target)
moves

[(13, 19),
 (19, 12),
 (6, 2),
 (2, 7),
 (16, 8),
 (7, 2),
 (2, 6),
 (8, 13),
 (3, 1),
 (12, 5),
 (6, 3),
 (1, 4),
 (5, 2),
 (13, 6)]

In [33]:
[(S.id_pos[x], S.id_pos[y]) for x, y in moves]

[((2, 3), (2, 2)),
 ((2, 2), (3, 2)),
 ((3, 3), (4, 3)),
 ((4, 3), (5, 3)),
 ((2, 5), (2, 4)),
 ((5, 3), (4, 3)),
 ((4, 3), (3, 3)),
 ((2, 4), (2, 3)),
 ((3, 4), (4, 4)),
 ((3, 2), (4, 2)),
 ((3, 3), (3, 4)),
 ((4, 4), (4, 5)),
 ((4, 2), (4, 3)),
 ((2, 3), (3, 3))]